# 04 — Engenharia de Features (Motor Geométrico)

Este notebook transforma os dados brutos das coordenadas (gerados no Módulo 1) em um **vetor numérico fixo e normalizado**, que é o formato exigido para treinar o modelo de Machine Learning no Módulo 3.

**Como funciona:**
1. **Cálculos Geométricos:** Mede distâncias e ângulos da trajetória entre a bola branca, as bolas alvo e as caçapas mais próximas.
2. **Normalização:** Converte todas as métricas para proporções (valores entre -1 e 1), ignorando a resolução real da tela.
3. **Zero-Padding:** Padroniza a entrada aplicando preenchimento de zeros para as bolas que já saíram da mesa, garantindo que o vetor final tenha sempre o tamanho exato de 107 colunas.

**Entrada:** `data/output/` (Arquivos `.json` com coordenadas e tipos)  
**Saída:** `data/dataset.csv` (Base de dados estruturada para o modelo)

In [ ]:
import os
import json
import numpy as np
import pandas as pd

## Célula 2 — O Motor Geométrico

Aqui definimos a classe `MotorGeometrico`. Ela encapsula toda a matemática do jogo:
* `calc_distancia`: Calcula a distância euclidiana.
* `calc_angulo`: Descobre a direção usando `arctan2`.
* `calc_angulo_corte`: Calcula a dificuldade da jogada (diferença entre o ângulo da branca e o ângulo para a caçapa).

Ao final, as bolas alvo são ordenadas da mais próxima à mais distante da branca.

In [ ]:
class MotorGeometrico:
    def __init__(self, largura=800.0, altura=400.0):
        # Usamos as dimensões padronizadas da mesa geradas no Módulo 1
        self.largura = largura
        self.altura = altura
        self.diag_max = np.hypot(largura, altura)
        
    def calc_distancia(self, p1, p2):
        # Calcula distância euclidiana
        return np.hypot(p2[0] - p1[0], p2[1] - p1[1])
    
    def calc_angulo(self, p1, p2):
        # Calcula o ângulo da trajetória
        return np.arctan2(p2[1] - p1[1], p2[0] - p1[0])
    
    def calc_angulo_corte(self, ang_branca_bola, ang_bola_cacapa):
        # Diferença de ângulo entre a batida e a caçapa
        diff = (ang_bola_cacapa - ang_branca_bola + np.pi) % (2 * np.pi) - np.pi
        return abs(diff)

    def processar_estado(self, dados_json):
        cacapas = [(c['x'], c['y']) for c in dados_json.get('cacapas', [])]
        bolas = dados_json.get('bolas', [])
        
        # Encontra a bola branca
        branca = next((b for b in bolas if b['tipo'] == 'branca'), None)
        outras_bolas = [b for b in bolas if b['tipo'] != 'branca']
        
        if not branca:
            # Se a branca caiu ou não foi detectada, retorna tudo zero
            return np.zeros(107)
            
        p_branca = (branca['x'], branca['y'])
        
        # Posição da branca normalizada (dividida pelo tamanho da mesa)
        features_branca = [branca['x'] / self.largura, branca['y'] / self.altura]
        
        lista_features_bolas = []
        
        for bola in outras_bolas:
            p_bola = (bola['x'], bola['y'])
            
            # Converte texto para número: lisa = 1, listrada = -1
            tipo_num = 1.0 if bola['tipo'] == 'lisa' else -1.0
            
            # Cálculos em relação à bola branca
            dist_w_b = self.calc_distancia(p_branca, p_bola)
            ang_w_b = self.calc_angulo(p_branca, p_bola)
            
            # Encontra qual é a caçapa mais perto dessa bola
            distancias_cacapas = [self.calc_distancia(p_bola, c) for c in cacapas]
            idx_cacapa_prox = np.argmin(distancias_cacapas)
            p_cacapa_prox = cacapas[idx_cacapa_prox]
            
            dist_b_p = distancias_cacapas[idx_cacapa_prox]
            ang_b_p = self.calc_angulo(p_bola, p_cacapa_prox)
            
            ang_corte = self.calc_angulo_corte(ang_w_b, ang_b_p)
            
            # Agrupa os cálculos dessa bola específica (valores entre 0 e 1)
            features_bola = [
                tipo_num,
                bola['x'] / self.largura,
                bola['y'] / self.altura,
                dist_w_b / self.diag_max,           
                ang_w_b / np.pi,                    
                dist_b_p / self.diag_max,           
                ang_corte / np.pi                   
            ]
            lista_features_bolas.append((dist_w_b, features_bola))
            
        # Ordena as bolas da mais próxima da branca para a mais distante
        lista_features_bolas.sort(key=lambda x: x[0])
        vetor_bolas = [f[1] for f in lista_features_bolas]
        
        # O PULO DO GATO: Se tiver menos de 15 bolas alvo, preenchemos com zeros
        while len(vetor_bolas) < 15:
            vetor_bolas.append([0.0] * 7)
            
        # Junta a branca com as 15 bolas em uma única lista reta de 107 itens
        vetor_final = np.concatenate(([features_branca], vetor_bolas), axis=None)
        return vetor_final

## Célula 3 — Execução e Exportação para o Dataset

Nesta etapa, o código carrega o JSON exportado pela detecção, aciona o `MotorGeometrico` para gerar o vetor fixo e adiciona essa linha de dados ao arquivo `dataset.csv`.

Se o arquivo CSV não existir, ele é criado junto com as colunas nomeadas. Se já existir, a nova linha é simplesmente anexada ao final (`mode='a'`), permitindo que você processe dezenas de imagens e acumule todas no mesmo arquivo.

In [ ]:
# 1. Definir caminhos relativos
# Presumindo que este notebook está em uma pasta ao lado de 'data'
caminho_json = os.path.join('..', 'data', 'output', 'IMG_0640_balls.json')
caminho_csv = os.path.join('..', 'data', 'dataset.csv')

# 2. Ler o JSON
try:
    with open(caminho_json, 'r', encoding='utf-8') as f:
        dados_da_mesa = json.load(f)
        
    print(f"Arquivo {os.path.basename(caminho_json)} carregado com sucesso!")
except FileNotFoundError:
    print(f"Erro: Arquivo não encontrado no caminho {caminho_json}")
    print("Verifique se as pastas estão escritas corretamente.")
    dados_da_mesa = None

# 3. Processar os dados se o JSON foi carregado
if dados_da_mesa:
    motor = MotorGeometrico()
    
    # Gera a linha com os 107 números
    vetor = motor.processar_estado(dados_da_mesa)
    
    # 4. Salvar em formato de tabela (CSV) para o Módulo 3
    colunas = ['branca_x', 'branca_y']
    for i in range(1, 16):
        colunas.extend([f'b{i}_tipo', f'b{i}_x', f'b{i}_y', f'b{i}_dist_branca', 
                        f'b{i}_ang_branca', f'b{i}_dist_cacapa', f'b{i}_ang_corte'])
        
    df = pd.DataFrame([vetor], columns=colunas)
    
    # Verifica existência do CSV para decidir entre criar ou apensar
    if not os.path.exists(caminho_csv):
        df.to_csv(caminho_csv, index=False)
        print(f"Novo dataset criado: {caminho_csv}")
    else:
        df.to_csv(caminho_csv, mode='a', header=False, index=False)
        print(f"Novo registro (frame) adicionado ao arquivo: {caminho_csv}")
        
    print(f"Resumo: Vetor extraído com {vetor.shape[0]} colunas.")